# Notebook 06 — LLM Narrative Engine

Membangun **LLM Narrative Engine** menggunakan **Groq API** yang mengubah output model menjadi laporan naratif otomatis Bahasa Indonesia.


> **Catatan Implementasi:** Di notebook ini fungsi disebut `build_crisis_context()`, sedangkan di `narrative_engine.py` (yang diimport dashboard) fungsinya bernama `build_context()`. Keduanya fungsional equivalen — perbedaan nama ini intentional: versi notebook memakai nama deskriptif, versi modul memakai nama pendek untuk API dashboard.

## 1. Import & Instalasi

In [ ]:
import pandas as pd
import numpy as np
import joblib
import json
import os
import warnings
warnings.filterwarnings('ignore')

try:
    from groq import Groq
    print('✓ groq SDK tersedia')
except ImportError:
    os.system('pip install groq -q')
    from groq import Groq
    print('✓ groq SDK berhasil diinstall')

print(f'Pandas: {pd.__version__}')


## 2. Konfigurasi API Key

In [ ]:
from dotenv import load_dotenv
import os
from groq import Groq

# Load dari file .env
load_dotenv()

GROQ_API_KEY = os.getenv('GROQ_API_KEY', '')
GROQ_MODEL = os.getenv(
    "GROQ_MODEL",
    "openai/gpt-oss-120b"
)

if not GROQ_API_KEY:
    print('API key belum diset!')
    print('   Dapatkan gratis di: https://console.groq.com/keys')
else:
    client = Groq(api_key=GROQ_API_KEY)
    print('✓ Groq client siap')
    print(f'   Key: {GROQ_API_KEY[:12]}...')

## 3. Load Data & Model

In [ ]:
predictions = pd.read_csv('../data/final/predictions_final.csv')
print('✓ predictions_final:', predictions.shape)
print('   Kolom:', predictions.columns.tolist())
print()

master = pd.read_parquet('../data/final/master_dataset_clean.parquet')
print('✓ master_dataset_clean:', master.shape)
print()

scaler   = joblib.load('../models/scaler.pkl')
rf_model = joblib.load('../models/model_random_forest.pkl')
le       = joblib.load('../models/label_encoder.pkl')
print('✓ Model artifacts loaded')
print()

print('=== 6 BULAN TERAKHIR ===')
print(predictions.tail(6)[['month','wisman','crisis_score_100',
    'crisis_level','rf_predicted_level','rf_confidence','iso_anomaly']].to_string())

## 4. Fungsi Build Context

In [ ]:
def build_crisis_context(month_str: str, narratives_cache: dict = None) -> dict:
    """
    Bangun context dict lengkap dari satu bulan untuk dikirim ke LLM, termasuk:
    - last_month_summary: ringkasan narasi bulan lalu (narrative consistency)
    - score_delta       : perubahan crisis score vs bulan lalu
    - dominant_factor   : faktor mana yang paling besar kontribusinya
    """
    pred_row = predictions[predictions['month'] == month_str]
    if len(pred_row) == 0:
        pred_row = predictions.tail(1)
        month_str = pred_row['month'].values[0]
    pred_row = pred_row.iloc[0]

    idx     = predictions[predictions['month'] == month_str].index[0]
    history = predictions.loc[max(0, idx-3):idx-1]

    # ── Context dasar ────────────────────────────────────────
    ctx = {
        'month'        : month_str,
        'crisis_score' : round(float(pred_row['crisis_score_100']), 1),
        'crisis_level' : str(pred_row.get('crisis_level', 'WASPADA')),
        'rf_predicted' : str(pred_row.get('rf_predicted_level', 'N/A')),
        'rf_confidence': round(float(pred_row.get('rf_confidence', 0)) * 100, 1),
        'is_anomaly'   : int(pred_row.get('iso_anomaly', 0)),
        'wisman'       : int(pred_row.get('wisman', 0)),
        'tpk_bintang'  : round(float(pred_row.get('tpk_bintang', 0)), 1),
        'inflasi'      : round(float(pred_row.get('inflasi_processed', 0)), 2),
        'usd_idr'      : round(float(pred_row.get('usd_idr_avg', 0)), 0),
        'sentiment'    : round(float(pred_row.get('avg_sentiment_monthly', 0)), 3),
        'prob_krisis'  : round(float(pred_row.get('prob_krisis', 0)) * 100, 1),
        'prob_siaga'   : round(float(pred_row.get('prob_siaga', 0)) * 100, 1),
        'bali_share'   : round(float(pred_row.get('bali_share_pct', 0)), 1),
        'wisman_zscore': round(float(pred_row.get('wisman_zscore', 0)), 2),
    }

    # ── Histori 3 bulan ──────────────────────────────────────
    if len(history) >= 2:
        avg3 = history['wisman'].mean()
        ctx['wisman_trend']  = 'naik' if ctx['wisman'] > avg3 else 'turun'
        ctx['avg_wisman_3m'] = round(avg3, 0)
        ctx['prev_levels']   = history['crisis_level'].tolist()
    else:
        ctx['wisman_trend']  = 'tidak tersedia'
        ctx['avg_wisman_3m'] = 0
        ctx['prev_levels']   = []

    # ── Score delta vs bulan lalu ─────────────────────────────
    if len(history) >= 1:
        prev_score = float(history.iloc[-1]['crisis_score_100'])
        ctx['score_delta'] = round(ctx['crisis_score'] - prev_score, 1)
        ctx['score_trend'] = 'MENINGKAT' if ctx['score_delta'] > 2 else (
                             'MENURUN'   if ctx['score_delta'] < -2 else 'STABIL')
    else:
        ctx['score_delta'] = 0.0
        ctx['score_trend'] = 'STABIL'

    # ── Dominant factor ───────────────────────────────────────
    factors = {
        'Kunjungan Wisatawan': abs(float(pred_row.get('wisman_zscore', 0))),
        'Tekanan Kurs'       : abs(float(pred_row.get('usd_volatility_3m', 0))) / 500.0,
        'Sentimen Negatif'   : float(pred_row.get('pct_negative_monthly', 0)) / 100.0
                               if 'pct_negative_monthly' in pred_row.index else 0.0,
    }
    ctx['dominant_factor'] = max(factors, key=factors.get)

    # ── Anomaly explanation ───────────────────────────────────
    zscore = ctx['wisman_zscore']
    if zscore <= -3:
        ctx['anomaly_explanation'] = (
            f'Z-score {zscore:.1f} -> kunjungan {abs(zscore):.1f} std di bawah rata-rata 12 bulan. '
            'Ini adalah kejadian ekstrem (<0.1% probabilitas pada distribusi normal).')
    elif zscore <= -2:
        ctx['anomaly_explanation'] = (
            f'Z-score {zscore:.1f} -> anomali signifikan, kunjungan jauh di bawah baseline historis.')
    else:
        ctx['anomaly_explanation'] = (
            f'Z-score {zscore:.1f} -> kunjungan masih dalam rentang normal.')

    # ── Narrative consistency: ringkasan bulan lalu dari cache ─
    ctx['last_month_summary'] = ''
    if narratives_cache and len(history) >= 1:
        prev_month = history.iloc[-1]['month']
        if prev_month in narratives_cache:
            prev_narrative = narratives_cache[prev_month].get('narrative', '')
            # Ambil hanya kalimat pertama sebagai memory anchor
            ctx['last_month_summary'] = prev_narrative.split('.')[0] + '.' if prev_narrative else ''

    return ctx

# Test dengan bulan terbaru
# test_month didefinisikan di sini — wajib ada sebelum cell generate di bawah
test_month = predictions['month'].iloc[-1]
ctx_test   = build_crisis_context(test_month)
print(f'Context test ({test_month}):')
for k, v in ctx_test.items():
    print(f'  {k:22s}: {v}')


## 5. Fungsi Build Prompt

In [ ]:
LEVEL_DESC = {
    'AMAN'    : 'kondisi pariwisata normal, tidak ada indikasi krisis',
    'WASPADA' : 'ada sinyal awal yang perlu dipantau',
    'SIAGA'   : 'tekanan signifikan pada sektor pariwisata',
    'KRISIS'  : 'krisis pariwisata yang membutuhkan respons segera'
}

def build_prompt(ctx: dict, report_type: str = 'summary') -> str:
    level_text = LEVEL_DESC.get(ctx['crisis_level'], ctx['crisis_level'])
    prev = ' -> '.join(ctx['prev_levels']) if ctx['prev_levels'] else 'N/A'

    # ── Data block utama ──────────────────────────────────────
    data_block = (
        f"DATA PARIWISATA BALI — {ctx['month']}\n"
        f"Crisis Score: {ctx['crisis_score']}/100 -> Level {ctx['crisis_level']} ({level_text})\n"
        f"Prediksi RF: {ctx['rf_predicted']} (confidence: {ctx['rf_confidence']}%)\n"
        f"Anomali: {'Ya' if ctx['is_anomaly'] else 'Tidak'} | "
        f"P(Krisis): {ctx['prob_krisis']}% | P(Siaga): {ctx['prob_siaga']}%\n"
        f"Wisman: {ctx['wisman']:,.0f} (trend: {ctx['wisman_trend']}, "
        f"avg 3bln: {int(ctx['avg_wisman_3m']):,.0f})\n"
        f"Pangsa Bali/Nasional: {ctx['bali_share']}% | Z-score: {ctx['wisman_zscore']}\n"
        f"TPK Hotel: {ctx['tpk_bintang']}% | USD/IDR: Rp {int(ctx['usd_idr']):,}\n"
        f"Inflasi: {ctx['inflasi']}% | Sentimen: {ctx['sentiment']}\n"
        f"Histori 3 bulan: {prev}\n"
    )

    # ── Comparative & causal block ─────────────────────
    comparative_block = (
        f"\nANALISIS KOMPARATIF:\n"
        f"Tren Crisis Score: {ctx['score_trend']} "
        f"({ctx['score_delta']:+.1f} poin vs bulan lalu)\n"
        f"Faktor dominan: {ctx['dominant_factor']}\n"
        f"Penjelasan anomali: {ctx['anomaly_explanation']}\n"
    )

    # ── Narrative consistency block ────────────────────
    continuity_block = ''
    if ctx.get('last_month_summary'):
        continuity_block = (
            f"\nKONTEKS BULAN LALU:\n"
            f"{ctx['last_month_summary']}\n"
            f"(Gunakan ini sebagai referensi perbandingan — bukan diulang kembali)\n"
        )

    full_data = data_block + comparative_block + continuity_block

    if report_type == 'summary':
        return (
            "Kamu adalah analis sistem BaliGuard (early warning pariwisata Bali).\n"
            + full_data
            + f"\nBuat ringkasan kondisi pariwisata Bali bulan {ctx['month']} "
            "dalam 2-3 kalimat Bahasa Indonesia. Sertakan: "
            "(1) status & skor, (2) satu angka kunci sebagai bukti, "
            "(3) arah tren vs bulan lalu. Tajam, berbasis data, cocok untuk KPI card."
        )
    elif report_type == 'alert':
        return (
            "Kamu adalah sistem BaliGuard. Kondisi kritis terdeteksi.\n"
            + full_data
            + "\nBuat PERINGATAN DARURAT Bahasa Indonesia (max 120 kata):\n"
            "1. Status level dan crisis score\n"
            "2. 3 indikator kritis yang MENYEBABKAN kondisi ini (sebab-akibat, bukan hanya angka)\n"
            "3. 1 rekomendasi tindakan segera yang konkret dan bisa dieksekusi dalam 7 hari\n"
            "Tone profesional, urgen, berbasis data."
        )
    else:
        return (
            "Kamu adalah analis BaliGuard.\n"
            + full_data
            + "\nBuat laporan bulanan Bahasa Indonesia dengan REASONING yang kuat:\n"
            "1. **Ringkasan Eksekutif** (2-3 kalimat): status, skor, perbandingan bulan lalu\n"
            "2. **Analisis Indikator** (3-4 kalimat): apa yang ditunjukkan angka-angka, "
            "mengapa setiap indikator PENTING untuk disimpulkan\n"
            "3. **Analisis Kausal** (2-3 kalimat): APA yang menyebabkan kondisi ini — "
            "bukan hanya deskripsi angka, tapi hubungan sebab-akibat antar indikator\n"
            "4. **Rekomendasi** (3 poin konkret dengan target waktu): "
            "setiap poin harus spesifik, actionable, dan berbasis data di atas\n"
            "Setiap klaim harus didukung minimal satu angka dari data."
        )

print(f'Summary prompt : {len(build_prompt(ctx_test, "summary"))} karakter')
print(f'Alert prompt   : {len(build_prompt(ctx_test, "alert"))} karakter')
print(f'Monthly prompt : {len(build_prompt(ctx_test, "monthly"))} karakter')
print()
print('=== SAMPLE MONTHLY PROMPT (200 char) ===')
print(build_prompt(ctx_test, 'monthly')[:200])


## 6. Fungsi Generate Narasi 

In [ ]:
def generate_narrative(month_str: str, report_type: str = 'summary',
                       api_key: str = None) -> dict:
    if api_key is None:
        api_key = os.getenv('GROQ_API_KEY', '')
    if not api_key or api_key.startswith('gsk_GANTI'):
        return {'success'  : False,
                'narrative': '[API key belum dikonfigurasi. Set GROQ_API_KEY.]',
                'error'    : 'No API key'}
    try:
        ctx      = build_crisis_context(month_str)
        prompt   = build_prompt(ctx, report_type)
        client   = Groq(api_key=api_key)
        response = client.chat.completions.create(
            model = GROQ_MODEL,
            messages    = [{'role': 'user', 'content': prompt}],
            temperature = 0.7,
            max_tokens  = 1024
        )
        return {
            'success'      : True,
            'narrative'    : response.choices[0].message.content,
            'tokens'       : response.usage.prompt_tokens + response.usage.completion_tokens,
            'month'        : month_str,
            'crisis_level' : ctx['crisis_level'],
            'report_type'  : report_type,
            'crisis_score' : ctx['crisis_score'],
        }
    except Exception as e:
        return {'success': False, 'narrative': f'[Error: {e}]', 'error': str(e)}

# Test
# Fallback: pastikan test_month terdefinisi meski cell sebelumnya di-skip
if 'test_month' not in dir():
    test_month = predictions['month'].iloc[-1]
print(f'Testing generate untuk bulan: {test_month}')
result = generate_narrative(test_month, 'summary', GROQ_API_KEY)
print(f'Success: {result["success"]}')
if result['success']:
    print(f'Tokens : {result["tokens"]}')
    print()
    print('=== NARASI ===')
    print(result['narrative'])
else:
    print(f'Error  : {result.get("error")}')


## 7. Batch Generate 

In [ ]:
crisis_months = predictions[
    predictions['crisis_level'].isin(['KRISIS', 'SIAGA'])
].sort_values('crisis_score_100', ascending=False)

print(f'Bulan KRISIS/SIAGA: {len(crisis_months)}')
narratives_cache = {}

if not GROQ_API_KEY.startswith('gsk_GANTI'):
    top_months = crisis_months.head(10)['month'].tolist()
    print(f'Generating {len(top_months)} narasi...')
    print()

    for i, month in enumerate(top_months, 1):
        level = predictions[predictions['month']==month]['crisis_level'].values[0]
        rtype = 'alert' if level == 'KRISIS' else 'monthly'
        print(f'  [{i:2d}/{len(top_months)}] {month} ({level})...', end='', flush=True)
        result = generate_narrative(month, rtype, GROQ_API_KEY)
        if result['success']:
            narratives_cache[month] = result
            print(f' ✓ {result["tokens"]} tokens')
        else:
            print(f' ✗ {result["error"]}')

    os.makedirs('../data/final', exist_ok=True)
    with open('../data/final/narratives_cache.json', 'w', encoding='utf-8') as f:
        json.dump(narratives_cache, f, ensure_ascii=False, indent=2)
    print(f'\n✓ narratives_cache.json disimpan ({len(narratives_cache)} narasi)')
else:
    print('⚠ API key belum diset — skip batch generation')


## 8. Export `src/narrative_engine.py`

In [ ]:
import os
os.makedirs('../src', exist_ok=True)

# CATATAN: Sejak patch NB05, 'pct_negative_monthly' dan 'usd_volatility_3m'
# sudah tersimpan di predictions_final.csv. Engine ini tidak perlu fallback 0.
# Guard .get(..., 0) tetap dipertahankan untuk keamanan saat kolom hilang.
engine_code = "import os\nimport json\nimport numpy as np\nfrom groq import Groq\n\nGROQ_MODEL = os.getenv(\n    \"GROQ_MODEL\",\n    \"openai/gpt-oss-120b\"\n)\n\nLEVEL_DESC = {\n    'AMAN'    : 'kondisi pariwisata normal, tidak ada indikasi krisis',\n    'WASPADA' : 'ada sinyal awal yang perlu dipantau',\n    'SIAGA'   : 'tekanan signifikan pada sektor pariwisata',\n    'KRISIS'  : 'krisis pariwisata yang membutuhkan respons segera'\n}\n\ndef build_context(pred_row: dict, history_rows: list = None,\n                  narratives_cache: dict = None) -> dict:\n    ctx = {\n        'month'        : str(pred_row.get('month', 'N/A')),\n        'crisis_score' : round(float(pred_row.get('crisis_score_100', 0)), 1),\n        'crisis_level' : str(pred_row.get('crisis_level', 'WASPADA')),\n        'rf_predicted' : str(pred_row.get('rf_predicted_level', 'N/A')),\n        'rf_confidence': round(float(pred_row.get('rf_confidence', 0)) * 100, 1),\n        'is_anomaly'   : int(pred_row.get('iso_anomaly', 0)),\n        'wisman'       : int(pred_row.get('wisman', 0)),\n        'tpk_bintang'  : round(float(pred_row.get('tpk_bintang', 0)), 1),\n        'inflasi'      : round(float(pred_row.get('inflasi_processed', 0)), 2),\n        'usd_idr'      : round(float(pred_row.get('usd_idr_avg', 0)), 0),\n        'sentiment'    : round(float(pred_row.get('avg_sentiment_monthly', 0)), 3),\n        'prob_krisis'  : round(float(pred_row.get('prob_krisis', 0)) * 100, 1),\n        'prob_siaga'   : round(float(pred_row.get('prob_siaga', 0)) * 100, 1),\n        'bali_share'   : round(float(pred_row.get('bali_share_pct', 0)), 1),\n        'wisman_zscore': round(float(pred_row.get('wisman_zscore', 0)), 2),\n        'recovery_pct' : round(float(pred_row.get('wisman_recovery_pct', 0)), 1),\n    }\n    if history_rows and len(history_rows) >= 2:\n        avg3 = np.mean([r.get('wisman', 0) for r in history_rows[-3:]])\n        ctx['wisman_trend']  = 'naik' if ctx['wisman'] > avg3 else 'turun'\n        ctx['avg_wisman_3m'] = round(avg3, 0)\n        ctx['prev_levels']   = [r.get('crisis_level', 'N/A') for r in history_rows[-3:]]\n        prev_score           = float(history_rows[-1].get('crisis_score_100', ctx['crisis_score']))\n        ctx['score_delta']   = round(ctx['crisis_score'] - prev_score, 1)\n        ctx['score_trend']   = ('MENINGKAT' if ctx['score_delta'] > 2\n                                else 'MENURUN' if ctx['score_delta'] < -2 else 'STABIL')\n    else:\n        ctx['wisman_trend']  = 'tidak tersedia'\n        ctx['avg_wisman_3m'] = 0\n        ctx['prev_levels']   = []\n        ctx['score_delta']   = 0.0\n        ctx['score_trend']   = 'STABIL'\n    factors = {\n        'Kunjungan Wisatawan': abs(ctx['wisman_zscore']),\n        'Tekanan Kurs'       : float(pred_row.get('usd_volatility_3m', 0)) / 500.0,\n        'Sentimen Negatif'   : float(pred_row.get('pct_negative_monthly', 0)) / 100.0,\n    }\n    ctx['dominant_factor'] = max(factors, key=factors.get)\n    zscore = ctx['wisman_zscore']\n    if zscore <= -3:\n        ctx['anomaly_exp'] = (f'Z-score {zscore:.1f} \u2014 kunjungan {abs(zscore):.1f} std '\n                              'di bawah rata-rata 12 bulan (kejadian ekstrem <0.1%).')\n    elif zscore <= -2:\n        ctx['anomaly_exp'] = f'Z-score {zscore:.1f} \u2014 anomali signifikan.'\n    else:\n        ctx['anomaly_exp'] = f'Z-score {zscore:.1f} \u2014 dalam rentang normal.'\n    ctx['last_month_summary'] = ''\n    if narratives_cache and history_rows:\n        prev_month = str(history_rows[-1].get('month', ''))\n        if prev_month in narratives_cache:\n            txt = narratives_cache[prev_month].get('narrative', '')\n            ctx['last_month_summary'] = txt.split('.')[0] + '.' if txt else ''\n    return ctx\n\ndef build_prompt(ctx: dict, report_type: str = 'summary') -> str:\n    level_text = LEVEL_DESC.get(ctx['crisis_level'], ctx['crisis_level'])\n    prev = ' -> '.join(ctx.get('prev_levels', [])) or 'N/A'\n    data_block = (\n        f\"DATA PARIWISATA BALI \u2014 {ctx['month']}\\n\"\n        f\"Crisis Score: {ctx['crisis_score']}/100 \u2192 Level {ctx['crisis_level']} ({level_text})\\n\"\n        f\"Tren score: {ctx.get('score_trend','STABIL')} ({ctx.get('score_delta',0):+.1f} poin)\\n\"\n        f\"Prediksi RF: {ctx['rf_predicted']} (confidence: {ctx['rf_confidence']}%)\\n\"\n        f\"Anomali: {'Ya' if ctx['is_anomaly'] else 'Tidak'} | \"\n        f\"P(Krisis): {ctx['prob_krisis']}% | P(Siaga): {ctx['prob_siaga']}%\\n\"\n        f\"Wisman: {ctx['wisman']:,.0f} (trend: {ctx['wisman_trend']}, \"\n        f\"avg 3bln: {int(ctx['avg_wisman_3m']):,.0f}, recovery: {ctx.get('recovery_pct',0):.1f}%)\\n\"\n        f\"Faktor dominan: {ctx.get('dominant_factor','N/A')}\\n\"\n        f\"{ctx.get('anomaly_exp','')}\\n\"\n        f\"Pangsa Bali: {ctx['bali_share']}% | Z-score: {ctx['wisman_zscore']}\\n\"\n        f\"TPK Hotel: {ctx['tpk_bintang']}% | USD/IDR: Rp {int(ctx['usd_idr']):,}\\n\"\n        f\"Inflasi: {ctx['inflasi']}% | Sentimen: {ctx['sentiment']}\\n\"\n        f\"Histori: {prev}\\n\"\n    )\n    if ctx.get('last_month_summary'):\n        data_block += f\"Konteks bulan lalu: {ctx['last_month_summary']}\\n\"\n    if report_type == 'summary':\n        return (\"Kamu adalah analis sistem BaliGuard.\\n\" + data_block\n                + f\"Buat ringkasan 2-3 kalimat Bahasa Indonesia: \"\n                \"(1) status & skor, (2) satu angka kunci, (3) arah tren. Tajam, berbasis data.\")\n    elif report_type == 'alert':\n        return (\"Kamu adalah sistem BaliGuard. Kondisi kritis terdeteksi.\\n\" + data_block\n                + \"Buat PERINGATAN DARURAT Bahasa Indonesia (max 120 kata): \"\n                \"status, 3 indikator penyebab dengan analisis sebab-akibat, \"\n                \"1 rekomendasi segera yang konkret.\")\n    else:\n        return (\"Kamu adalah analis BaliGuard.\\n\" + data_block\n                + \"Buat laporan bulanan Bahasa Indonesia:\\n\"\n                \"1. Ringkasan Eksekutif (2-3 kalimat + perbandingan bulan lalu)\\n\"\n                \"2. Analisis Indikator (3-4 kalimat dengan angka)\\n\"\n                \"3. Analisis Kausal (2-3 kalimat: sebab-akibat, bukan deskripsi)\\n\"\n                \"4. Rekomendasi (3 poin konkret + target waktu, berbasis data)\\n\"\n                \"Setiap klaim harus dikuatkan minimal satu angka.\")\n\ndef generate(pred_row: dict, report_type: str = 'summary',\n             api_key: str = None, history_rows: list = None,\n             narratives_cache: dict = None) -> dict:\n    if api_key is None:\n        api_key = os.getenv('GROQ_API_KEY', '')\n    if not api_key:\n        return {'success': False, 'narrative': '[API key tidak dikonfigurasi]',\n                 'error': 'No API key'}\n    try:\n        ctx      = build_context(pred_row, history_rows, narratives_cache)\n        client   = Groq(api_key=api_key)\n        response = client.chat.completions.create(\n            model       = GROQ_MODEL,\n            messages    = [{'role': 'user', 'content': build_prompt(ctx, report_type)}],\n            temperature = 0.7,\n            max_tokens  = 1024\n        )\n        return {\n            'success'      : True,\n            'narrative'    : response.choices[0].message.content,\n            'tokens'       : response.usage.prompt_tokens + response.usage.completion_tokens,\n            'month'        : ctx['month'],\n            'crisis_level' : ctx['crisis_level'],\n        }\n    except Exception as e:\n        return {'success': False, 'narrative': f'[Error: {e}]', 'error': str(e)}\n\ndef load_cache(cache_path: str = '../data/final/narratives_cache.json') -> dict:\n    if os.path.exists(cache_path):\n        with open(cache_path, 'r', encoding='utf-8') as f:\n            return json.load(f)\n    return {}\n"

with open('../src/__init__.py', 'w') as f:
    f.write('')

with open('../src/narrative_engine.py', 'w', encoding='utf-8') as f:
    f.write(engine_code)

print('../src/narrative_engine.py disimpan')
print(f'   Model   : {GROQ_MODEL}')
print(f'   Fitur   : score_delta, dominant_factor, anomaly_exp, recovery_pct, last_month_summary')
print(f'   Fungsi  : build_context(), build_prompt(), generate(), load_cache()')
print()
print('Import di dashboard:')
print('  from src.narrative_engine import generate, build_context, load_cache')


## 9. Checklist File

In [ ]:
print('=' * 55)
print('  BALIGUARD — RINGKASAN NB06')
print('=' * 55)
print()
required = [
    '../data/final/master_dataset_clean.parquet',
    '../data/final/predictions_final.csv',
    '../data/final/narratives_cache.json',
    '../models/model_random_forest.pkl',
    '../models/model_isolation_forest.pkl',
    '../models/scaler.pkl',
    '../models/label_encoder.pkl',
    '../src/narrative_engine.py',
]
for f in required:
    status = '✓' if os.path.exists(f) else '✗ belum ada'
    print(f'   {status} {f}')
print()
print('⚠ PENTING BACA INI DULU ⚠')
print('Jalankan dashboard:')
print('  pip install streamlit plotly groq pyarrow joblib python-dotenv')
print('  python update_pipeline.py')
print('  streamlit run dashboard.py')
print('=' * 55)